# A-001 Enterprise Assistant — Two Tools with LlamaIndex
## Asset 360 + Project Risk 360 + Existing A001 Vector Store + Gradio

This notebook is the **next step** after the simple SQL + RAG notebook.

### What changes in this version?

We keep the design easy for a cohort:

**Tool 1 — Enterprise SQL Tool**
- reads `workspace.nuclear_enterprise_360.asset_360`
- reads `workspace.nuclear_enterprise_360.project_risk_360`
- can use either view or join both
- only `SELECT` is allowed

**Tool 2 — A001 Document RAG Tool**
- uses **LlamaIndex**
- reloads the **existing persisted A001 vector store**
- uses the same Qwen embedding model for query embeddings
- retrieves the most relevant document chunks

### Flow

`Question`
→ `SQL Tool: Asset 360 + Project Risk 360`
→ `RAG Tool: existing A001 vector store`
→ `GPT-OSS 120B synthesis`
→ `Grounded answer`

### Important training note

All A-001 data and documents in this project are synthetic training material.

## 0. Before you run

This notebook expects:

```text
Asset view:
workspace.nuclear_enterprise_360.asset_360

Project/risk view:
workspace.nuclear_enterprise_360.project_risk_360

Existing LlamaIndex store:
/Volumes/workspace/nuclear_enterprise_360/training_files/a001_rag_store
```

The notebook **does not rebuild the PDF embeddings**.  
It reloads the vector store that was created in the earlier RAG notebook.

Run the cells from top to bottom.

## 1. Imports and configuration

For learners, there are only a few important settings:

- `ASSET_VIEW` — current asset-level structured facts.
- `PROJECT_RISK_VIEW` — project/risk/action facts.
- `PERSIST_DIR` — the existing LlamaIndex vector store.
- `CHAT_MODEL` — writes SQL and the final response.
- `EMBED_MODEL` — embeds the user's question for semantic retrieval.

In [0]:
%pip install -q -U \
    "typing_extensions>=4.15.0,<5" \
    "gradio>=6,<7" \
    llama-index \
    llama-index-llms-databricks \
    llama-index-embeddings-databricks

dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# CELL 2 — Imports + configuration

import json
import re
from pathlib import Path

import gradio as gr

from llama_index.core import Settings, StorageContext, load_index_from_storage
from llama_index.core.tools import FunctionTool
from llama_index.embeddings.databricks import DatabricksEmbedding
from llama_index.llms.databricks import Databricks


# ------------------------------------------------------------
# DATA SOURCES
# ------------------------------------------------------------

ASSET_VIEW = "workspace.nuclear_enterprise_360.asset_360"
PROJECT_RISK_VIEW = "workspace.nuclear_enterprise_360.project_risk_360"

PERSIST_DIR = Path(
    "/Volumes/workspace/nuclear_enterprise_360/training_files/a001_rag_store"
)


# ------------------------------------------------------------
# MODELS
# ------------------------------------------------------------

CHAT_MODEL = "databricks-gpt-oss-120b"
EMBED_MODEL = "databricks-qwen3-embedding-0-6b"

TEMPERATURE = 0.1
TOP_K = 4
MAX_SQL_ROWS = 20


print("Configuration loaded")
print("Asset view       :", ASSET_VIEW)
print("Project/risk view:", PROJECT_RISK_VIEW)
print("Vector store     :", PERSIST_DIR)
print("Chat model       :", CHAT_MODEL)
print("Embedding model  :", EMBED_MODEL)

Configuration loaded
Asset view       : workspace.nuclear_enterprise_360.asset_360
Project/risk view: workspace.nuclear_enterprise_360.project_risk_360
Vector store     : /Volumes/workspace/nuclear_enterprise_360/training_files/a001_rag_store
Chat model       : databricks-gpt-oss-120b
Embedding model  : databricks-qwen3-embedding-0-6b


## 2. Configure LlamaIndex for Databricks

The earlier vector store already contains document vectors.

We still need the **same embedding model** at question time because the new question must be converted into a vector before LlamaIndex can compare it with the stored vectors.

This cell uses the current Databricks notebook's workspace URL and API token, so students do not have to paste a token manually.

In [0]:
# CELL 3 — LlamaIndex + Databricks model setup

API_ROOT = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiUrl()
    .get()
)

API_TOKEN = (
    dbutils.notebook.entry_point
    .getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

SERVING_ENDPOINT = f"{API_ROOT}/serving-endpoints"


# LLM used for:
# 1) generating safe SQL
# 2) writing the final grounded answer
llm = Databricks(
    model=CHAT_MODEL,
    api_key=API_TOKEN,
    api_base=SERVING_ENDPOINT,
    temperature=TEMPERATURE,
)


# Same embedding model used when the A001 vector store was created
embed_model = DatabricksEmbedding(
    model=EMBED_MODEL,
    api_key=API_TOKEN,
    endpoint=SERVING_ENDPOINT,
)


Settings.llm = llm
Settings.embed_model = embed_model

print("LlamaIndex models configured.")

LlamaIndex models configured.


## 3. Validate the two SQL views

Before asking an LLM to write SQL, show learners the real schemas.

This matters because the SQL generator should use **actual columns**, not guessed columns.

In [0]:
# CELL 4 — Validate both structured views and show schemas

def schema_text(view_name):
    """Return a simple readable schema for a Spark table/view."""
    return "\n".join(
        f"- {column}: {dtype}"
        for column, dtype in spark.table(view_name).dtypes
    )


# This fails early with a clear error if a view is missing.
spark.table(ASSET_VIEW).limit(1).collect()
spark.table(PROJECT_RISK_VIEW).limit(1).collect()

ASSET_SCHEMA = schema_text(ASSET_VIEW)
PROJECT_RISK_SCHEMA = schema_text(PROJECT_RISK_VIEW)

print("=" * 70)
print("ASSET 360 SCHEMA")
print("=" * 70)
print(ASSET_SCHEMA)

print("\n" + "=" * 70)
print("PROJECT RISK 360 SCHEMA")
print("=" * 70)
print(PROJECT_RISK_SCHEMA)

ASSET 360 SCHEMA
- asset_id: string
- asset_name: string
- asset_type: string
- criticality: string
- status: string
- system_name: string
- unit_id: string
- health_score: double
- risk_level: string
- recommended_action: string
- open_work_orders: bigint
- high_priority_open_work: bigint
- follow_up_findings: bigint
- latest_inspection_date: string

PROJECT RISK 360 SCHEMA
- project_id: string
- project_name: string
- project_status: string
- completion_pct: double
- budget_usd: double
- active_risks: bigint
- maximum_exposure: bigint
- active_actions: bigint
- overdue_actions: bigint


## 4. Load the existing A001 vector store

No PDFs are re-read here.

No document embeddings are rebuilt here.

We simply load the persisted LlamaIndex index and create a retriever.

In [0]:
# CELL 5 — Load the existing persisted LlamaIndex vector store

if not PERSIST_DIR.exists():
    raise FileNotFoundError(
        f"""
Vector store not found:

{PERSIST_DIR}

Run the earlier A001 persistent RAG notebook first,
or update PERSIST_DIR to the folder where that store was saved.
"""
    )

storage_context = StorageContext.from_defaults(
    persist_dir=str(PERSIST_DIR)
)

a001_index = load_index_from_storage(
    storage_context=storage_context,
    embed_model=embed_model,
)

a001_retriever = a001_index.as_retriever(
    similarity_top_k=TOP_K
)

print("Existing A001 vector store loaded.")
print("Top K:", TOP_K)

Existing A001 vector store loaded.
Top K: 4


# TOOL 1 — Enterprise SQL Tool

One SQL tool now covers **both** structured views.

The LLM is given both schemas and may:

- query only `asset_360`
- query only `project_risk_360`
- join the two views when the question needs both

The validator blocks write operations and blocks access to other tables.

In [0]:
# CELL 6 — SQL helper functions

ALLOWED_VIEWS = {
    ASSET_VIEW.lower(),
    PROJECT_RISK_VIEW.lower(),
}


def clean_sql(sql_text):
    """Remove markdown fences and trailing semicolon."""
    sql_text = str(sql_text).strip()

    sql_text = re.sub(
        r"^```(?:sql)?\s*",
        "",
        sql_text,
        flags=re.IGNORECASE,
    )

    sql_text = re.sub(
        r"\s*```$",
        "",
        sql_text,
        flags=re.IGNORECASE,
    )

    return sql_text.rstrip(";").strip()


def validate_sql(sql):
    """
    Safety rules for the classroom demo:
    - SELECT only
    - no data-changing commands
    - only the two approved views
    """
    upper = f" {sql.upper()} "

    if not sql.upper().startswith("SELECT"):
        raise ValueError("Only a SELECT statement is allowed.")

    forbidden_words = [
        " INSERT ",
        " UPDATE ",
        " DELETE ",
        " DROP ",
        " ALTER ",
        " CREATE ",
        " MERGE ",
        " TRUNCATE ",
        " REPLACE ",
    ]

    for word in forbidden_words:
        if word in upper:
            raise ValueError(f"Blocked SQL operation: {word.strip()}")

    # Capture every object immediately following FROM or JOIN.
    referenced_objects = re.findall(
        r"\b(?:FROM|JOIN)\s+([`A-Za-z0-9_.-]+)",
        sql,
        flags=re.IGNORECASE,
    )

    referenced_objects = {
        obj.replace("`", "").lower()
        for obj in referenced_objects
    }

    invalid = referenced_objects - ALLOWED_VIEWS

    if invalid:
        raise ValueError(
            "SQL tried to access a non-approved object: "
            + ", ".join(sorted(invalid))
        )


def llm_complete(prompt):
    """Convert the LlamaIndex LLM response into plain text."""
    response = llm.complete(prompt)
    return str(response).strip()

In [0]:
# CELL 7 — TOOL 1 function: SQL over Asset 360 + Project Risk 360

def enterprise_sql_tool(question: str) -> str:
    """
    Use structured enterprise data to answer questions about:
    asset health/status/work orders and project/risk/action information.
    """

    prompt = f"""
You are a Databricks Spark SQL generator.

You have ONLY these two approved views.

VIEW 1:
{ASSET_VIEW}

SCHEMA:
{ASSET_SCHEMA}

VIEW 2:
{PROJECT_RISK_VIEW}

SCHEMA:
{PROJECT_RISK_SCHEMA}

USER QUESTION:
{question}

Rules:
1. Return ONE Spark SQL SELECT statement only.
2. Use only the two approved views above.
3. Use one view if that is enough.
4. Join the two views only when the question needs both.
5. Never guess a column name: use only columns shown in the schemas.
6. Never modify data.
7. Prefer LIMIT {MAX_SQL_ROWS}.
8. Asset IDs may look like A-001.
9. No markdown.
10. No explanation.
"""

    sql = clean_sql(llm_complete(prompt))
    validate_sql(sql)

    rows = (
        spark.sql(sql)
        .limit(MAX_SQL_ROWS)
        .collect()
    )

    rows = [
        row.asDict(recursive=True)
        for row in rows
    ]

    result = {
        "sql": sql,
        "rows": rows,
    }

    return json.dumps(
        result,
        indent=2,
        default=str,
    )

# TOOL 2 — A001 Document RAG Tool

This tool does **retrieval only**.

It searches the existing persisted vector store and returns the top matching document chunks with useful metadata such as source, page, score, and approval status when available.

Keeping retrieval separate from final answer generation makes the flow much easier to explain.

In [0]:
# CELL 8 — TOOL 2 function: search the persisted A001 vector store

def a001_document_tool(question: str) -> str:
    """
    Retrieve relevant A001 document evidence from the existing LlamaIndex store.
    """

    results = a001_retriever.retrieve(question)

    evidence = []

    for rank, item in enumerate(results, start=1):
        node = item.node
        metadata = node.metadata or {}

        source = (
            metadata.get("file_name")
            or metadata.get("source")
            or metadata.get("document_id")
            or metadata.get("file_path")
            or "A001 document"
        )

        page = (
            metadata.get("page_label")
            or metadata.get("page_number")
            or metadata.get("page")
            or "unknown"
        )

        approval = (
            metadata.get("approval_status")
            or metadata.get("status")
            or "unknown"
        )

        score = (
            round(float(item.score), 4)
            if item.score is not None
            else None
        )

        text = node.get_content().strip()

        evidence.append(
            f"""
[{rank}]
SOURCE: {source}
PAGE: {page}
APPROVAL STATUS: {approval}
SIMILARITY: {score}
TEXT:
{text}
""".strip()
        )

    return "\n\n".join(evidence)

## 5. Register the two LlamaIndex tools

This is the key teaching point:

```text
TOOL 1 = structured data
TOOL 2 = unstructured documents
```

We use `FunctionTool` so each normal Python function becomes a LlamaIndex tool with a name and description.

In [0]:
from llama_index.core.tools import FunctionTool

In [0]:
# CELL 9 — Create exactly TWO LlamaIndex tools

sql_tool = FunctionTool.from_defaults(
    fn=enterprise_sql_tool,
    name="enterprise_sql",
    description=(
        "Read-only SQL tool for structured facts from Asset 360 "
        "and Project Risk 360. Use for current asset, project, risk, "
        "status, score, work-order, and action data."
    ),
)

rag_tool = FunctionTool.from_defaults(
    fn=a001_document_tool,
    name="a001_document_rag",
    description=(
        "Search the existing A001 LlamaIndex vector store for "
        "inspection, reliability, policy, procedure, and document evidence."
    ),
)

TOOLS = [sql_tool, rag_tool]

print("Two LlamaIndex tools ready:")
for tool in TOOLS:
    print("-", tool.metadata.name, ":", tool.metadata.description)

Two LlamaIndex tools ready:
- enterprise_sql : Read-only SQL tool for structured facts from Asset 360 and Project Risk 360. Use for current asset, project, risk, status, score, work-order, and action data.
- a001_document_rag : Search the existing A001 LlamaIndex vector store for inspection, reliability, policy, procedure, and document evidence.


## 6. Main assistant — call both tools, then synthesize

For a beginner cohort, we make orchestration explicit instead of hiding it inside another agent layer.

Every question follows the same visible sequence:

1. SQL tool tries to get structured facts.
2. RAG tool gets document evidence.
3. The LLM combines what the tools returned.
4. If one source has no evidence, the answer says so instead of inventing facts.

In [0]:
# CELL 10 — Main two-tool assistant

def tool_content(tool_output):
    """FunctionTool returns ToolOutput; this extracts its text safely."""
    if hasattr(tool_output, "content"):
        return tool_output.content
    return str(tool_output)


def ask(question: str, verbose: bool = True) -> str:

    question = (question or "").strip()

    if not question:
        return "Please enter a question."

    if verbose:
        print("=" * 72)
        print("QUESTION")
        print("=" * 72)
        print(question)

    # --------------------------------------------------------
    # TOOL 1 — STRUCTURED SQL
    # --------------------------------------------------------
    try:
        sql_output = sql_tool(question=question)
        sql_evidence = tool_content(sql_output)
    except Exception as exc:
        sql_evidence = f"SQL TOOL ERROR: {type(exc).__name__}: {exc}"

    if verbose:
        print("\n" + "=" * 72)
        print("TOOL 1 — ENTERPRISE SQL")
        print("=" * 72)
        print(sql_evidence)

    # --------------------------------------------------------
    # TOOL 2 — DOCUMENT RAG
    # --------------------------------------------------------
    try:
        rag_output = rag_tool(question=question)
        rag_evidence = tool_content(rag_output)
    except Exception as exc:
        rag_evidence = f"RAG TOOL ERROR: {type(exc).__name__}: {exc}"

    if verbose:
        print("\n" + "=" * 72)
        print("TOOL 2 — A001 DOCUMENT RAG")
        print("=" * 72)
        print(rag_evidence)

    # --------------------------------------------------------
    # FINAL SYNTHESIS
    # --------------------------------------------------------
    final_prompt = f"""
You are an enterprise A001 assistant.

USER QUESTION:
{question}

TOOL 1 — STRUCTURED SQL EVIDENCE:
{sql_evidence}

TOOL 2 — DOCUMENT RAG EVIDENCE:
{rag_evidence}

Answering rules:
1. Use only the evidence supplied by the two tools.
2. Structured SQL evidence is authoritative for current structured values.
3. For procedures or policy, prefer APPROVED/current documents.
4. Do not treat DRAFT or SUPERSEDED material as current authority.
5. If the evidence is missing or a tool failed, say that clearly.
6. Do not invent columns, values, projects, risks, procedures, or actions.
7. Mention document source/page when that metadata is available.
8. Keep the answer concise and easy for a business user to understand.
9. Finish with a short suggested human next step only when useful.
"""

    answer = llm_complete(final_prompt)

    if verbose:
        print("\n" + "=" * 72)
        print("FINAL ANSWER")
        print("=" * 72)
        print(answer)

    return answer

## 7. Cohort tests

Run these in order.

### Test A — Asset 360
Checks the first structured view.

### Test B — Project Risk 360
Checks the second structured view.

### Test C — Hybrid
Checks both structured data and document RAG together.

In [0]:
# CELL 11A — Test Asset 360

# Override llm_complete to handle reasoning model content blocks
# databricks-gpt-oss-120b returns content as a list, not a string
import requests as _requests


def llm_complete(prompt):
    """Call the LLM directly, extracting text from reasoning model content blocks."""
    url = f"{SERVING_ENDPOINT}/{CHAT_MODEL}/invocations"
    headers = {
        "Authorization": f"Bearer {API_TOKEN}",
        "Content-Type": "application/json",
    }
    payload = {
        "messages": [{"role": "user", "content": prompt}],
        "temperature": TEMPERATURE,
    }
    resp = _requests.post(url, headers=headers, json=payload)
    resp.raise_for_status()
    data = resp.json()
    content = data["choices"][0]["message"]["content"]
    if isinstance(content, list):
        return " ".join(
            block.get("text", "")
            for block in content
            if block.get("type") == "text"
        ).strip()
    return str(content).strip()


ask(
    ''' 
    For A-001, summarize the current asset condition and related project risks,
then tell me what the approved inspection or reliability documents say.
    ''',
    verbose=True,
)

QUESTION
For A-001, summarize the current asset condition and related project risks,
then tell me what the approved inspection or reliability documents say.

TOOL 1 — ENTERPRISE SQL
{
  "sql": "SELECT \n  a.asset_id,\n  a.asset_name,\n  a.asset_type,\n  a.criticality,\n  a.status,\n  a.system_name,\n  a.unit_id,\n  a.health_score,\n  a.risk_level,\n  a.recommended_action,\n  a.open_work_orders,\n  a.high_priority_open_work,\n  a.follow_up_findings,\n  a.latest_inspection_date,\n  p.project_id,\n  p.project_name,\n  p.project_status,\n  p.completion_pct,\n  p.budget_usd,\n  p.active_risks,\n  p.maximum_exposure,\n  p.active_actions,\n  p.overdue_actions\nFROM workspace.nuclear_enterprise_360.asset_360 AS a\nLEFT JOIN workspace.nuclear_enterprise_360.project_risk_360 AS p\n  ON a.asset_id = p.project_id\nWHERE a.asset_id = 'A-001'\nLIMIT 20",
  "rows": [
    {
      "asset_id": "A-001",
      "asset_name": "Pump 001",
      "asset_type": "Pump",
      "criticality": "HIGH",
      "status

'**Asset\u202fA‑001 (Pump\u202f001) – Current Condition (SQL evidence)**  \n\n| Item | Value |\n|------|-------|\n| **Asset ID / Name** | A‑001 – Pump\u202f001 |\n| **Type / System** | Pump, Cooling Water (unit\u202fTRN‑A) |\n| **Criticality** | HIGH |\n| **Status** | IN\u202fSERVICE |\n| **Health Score** | **68.05** (below peer‑average) |\n| **Risk Level** | **MEDIUM** |\n| **Recommended Action** | *Increase monitoring* |\n| **Open Work Orders** | 6 (3 are high‑priority) |\n| **Follow‑up Findings** | 2 pending |\n| **Latest Inspection Date** | 23\u202fAug\u202f2026 |\n\n**Related Project Risks** – The asset has **no linked project** (`project_id` is null), so there are **no active project‑level risks** recorded for A‑001 at this time.\n\n---\n\n### What the Approved Inspection / Reliability Documents Say  \n\n**Current Procedure:** *A001‑PROC‑INS‑001‑V2* (Approved) – see sources **[3]** (page\u202f2) and **[4]** (page\u202f1).  \n\nKey points:\n\n1. **Purpose** – Build a defensible co

In [0]:
# CELL 11B — Test Project Risk 360

ask(
    "For A-001, what project and risk information is available in Project Risk 360?",
    verbose=True,
)

In [0]:
# CELL 11C — Hybrid test: SQL + project/risk + document evidence

ask(
    """
For A-001, summarize the current asset condition and related project risks,
then tell me what the approved inspection or reliability documents say.
""",
    verbose=True,
)

## 8. Optional learner question box

Use this if you want learners to type a question directly inside the Databricks notebook without opening Gradio.

In [0]:
# CELL 12 — Databricks widget

dbutils.widgets.text(
    "enterprise_question",
    "For A-001, summarize the asset status, project risks, and approved inspection guidance."
)

user_question = dbutils.widgets.get("enterprise_question")

ask(
    user_question,
    verbose=True,
)

# 9. Simple Gradio UI

The UI is intentionally minimal.

The Gradio app only needs one function:

```python
gradio_chat(message, history)
```

The history is handled by Gradio.  
Our two-tool logic stays inside `ask()`.

In [0]:
# CELL 13 — Simple Gradio 6 chat UI

def gradio_chat(message, history):
    return ask(
        message,
        verbose=False,
    )


demo = gr.ChatInterface(
    fn=gradio_chat,
    title="A001 Enterprise Assistant",
    description=(
        "Ask about A-001 asset status, projects, risks, "
        "and approved document guidance. "
        "The assistant uses two tools: SQL + document RAG."
    ),
    examples=[
        "What is the current status of A-001?",
        "What project risks are related to A-001?",
        "What approved inspection guidance applies to A-001?",
        "Combine A-001 asset status, project risks, and approved inspection guidance.",
    ],
)


# Gradio 6 uses the messages format automatically.
# share=False is recommended for notebook use.
demo.launch(
    share=False,
    inline=True,
)

# 10. What to explain to the cohort

### The core idea

**Structured data and documents solve different parts of the question.**

| Source | Best for |
|---|---|
| Asset 360 | health score, risk level, work orders, asset status |
| Project Risk 360 | project, risk, action, project-level context |
| A001 vector store | procedures, policy, inspection, reliability evidence |

### Why two tools?

The learner should see a clean mental model:

```text
Question
  |
  +--> SQL Tool --> Asset 360 / Project Risk 360
  |
  +--> RAG Tool --> A001 vector store
  |
  +--> Final grounded answer
```

### Why not rebuild embeddings?

Because the earlier notebook already did the expensive indexing work.

A persistent vector store means:

```text
Build once
Persist once
Reload many times
```

### Why read-only SQL?

The assistant is for question answering.  
It should not update, delete, merge, or create enterprise data.

---

# Troubleshooting

### 1. `project_risk_360` not found

Check the views:

```sql
SHOW TABLES IN workspace.nuclear_enterprise_360;
```

The expected project/risk view for this training project is:

```text
project_risk_360
```

### 2. Vector store not found

Expected path:

```text
/Volumes/workspace/nuclear_enterprise_360/training_files/a001_rag_store
```

Run the earlier persistent RAG notebook first if the store has not been created.

### 3. Embedding/model endpoint error

Expected embedding endpoint:

```text
databricks-qwen3-embedding-0-6b
```

Expected chat endpoint in this notebook:

```text
databricks-gpt-oss-120b
```

If your workspace exposes a different chat endpoint, change `CHAT_MODEL` in the configuration cell.

### 4. SQL generation error

The notebook prints the generated SQL in the Tool 1 evidence.

The generator is deliberately restricted to:

```text
workspace.nuclear_enterprise_360.asset_360
workspace.nuclear_enterprise_360.project_risk_360
```

This keeps the classroom example safe and understandable.